In [ ]:
import numpy as np
import mygene

## we use geneformer foundation model as backbone llm to extract foundation gene embeddings
from geneformer.tokenizer import TranscriptomeTokenizer
from datasets import load_from_disk

import os
import pandas as pd
import anndata as ad
from scipy import sparse
import torch
from transformers import AutoModel

In [ ]:
selected_genes = np.genfromtxt('/preprocess_step0_output_dir/hmhvg_gene_list.txt', dtype=str)

mg = mygene.MyGeneInfo()
symbols = list(selected_genes)

# （species='human'），return ensembl.gene_id
res = mg.querymany(symbols, scopes='symbol', fields='ensembl.gene', species='human')

symbol_to_ens = {}
for r in res:
    if 'notfound' in r and r['notfound']:
        continue
    sym = r['query']
    if 'ensembl' in r:
        ens_field = r['ensembl']
        if isinstance(ens_field, list):
            ens_id = ens_field[0]['gene']
        else:
            ens_id = ens_field['gene']
        symbol_to_ens[sym] = ens_id

ens_to_symbol = {}
for (gene, ens) in symbol_to_ens.items():
    ens_to_symbol[ens] = gene

In [ ]:
# choose model：V2(4096, with CLS) or V1(2048, without CLS)
tk = TranscriptomeTokenizer(model_version="V2") 
#{Ensembl_ID: token_id}
vocab = {k.split('.')[0]: v for k, v in tk.gene_token_dict.items()}

def in_vocab(symbol):
    ens = symbol_to_ens.get(symbol)
    if not ens: 
        return symbol, None, False, None
    ens_core = str(ens).split('.')[0]
    return symbol, ens_core, (ens_core in vocab), vocab.get(ens_core)

rows = [in_vocab(sym) for sym in selected_genes]

In [ ]:
def build_vocab_kept_index(rows, selected_genes):
    """
    from rows to extract genes: in_vocab=True：
    1) sym -> ens 
    2) sym -> original expr index
    return：kept_cols、ens_ids、sym_of_ens
    """
    # rows: (symbol, ensembl, in_vocab, token_id)
    sym2ens = {}
    for sym, ens, ok, _ in rows:
        if ok and ens:
            sym2ens[sym] = str(ens).split('.')[0]

    # symbol -> index
    sym2idx = {sym: j for j, sym in enumerate(selected_genes)}
    kept_cols, ens_ids, sym_of_ens, genes = [], [], {}, []
    for sym, ens_core in sym2ens.items():
        if sym in sym2idx:
            kept_cols.append(sym2idx[sym])
            ens_ids.append(ens_core)
            sym_of_ens[ens_core] = sym 
            genes.append(sym)
    return kept_cols, ens_ids, sym_of_ens, genes

kept_cols, ens_ids, sym_of_ens, encode_genes = build_vocab_kept_index(rows, selected_genes)

In [ ]:
geneformer_symbol_to_ens = {}
geneformer_ens_to_symbol = {}
for ens in ens_ids:
    geneformer_symbol_to_ens[ens_to_symbol[ens]] = ens
    geneformer_ens_to_symbol[ens] = ens_to_symbol[ens]

save_gene_list = list(geneformer_symbol_to_ens.keys())
print(len(save_gene_list))

with open(os.path.join('/preprocess_step0_output_dir', f"hmhvg_geneformer_gene_list.txt"), "w") as f:
        for gene in save_gene_list:
            f.write(gene + "\n")

In [ ]:
def make_minimal_adata(expr, ens_ids, barcodes):
    """make Geneformer h5ad：var['ensembl_id']、obs['n_counts']"""
    X = expr.copy()
    X = sparse.csr_matrix(X) if not sparse.issparse(X) else X
    obs = pd.DataFrame(index=[str(b) for b in barcodes])
    obs["n_counts"] = np.asarray(X.sum(axis=1)).ravel().astype(np.int64)
    obs["barcode"] = obs.index
    var = pd.DataFrame(index=[f"g{i}" for i in range(len(ens_ids))])
    var["ensembl_id"] = ens_ids
    return ad.AnnData(X=X, obs=obs, var=var)

In [ ]:
wsi_id = 'your_wsi_id'   
st_path = os.path.join('/path/to/your/st', wsi_id + '.h5ad')
adata = ad.read_h5ad(st_path)
barcodes = adata.obs_names
print(len(barcodes))
expr = adata[:, encode_genes].X.toarray()
adata_min = make_minimal_adata(expr, ens_ids, barcodes)

In [ ]:
def write_and_tokenize(adata_min, 
                       model_version="V2",
                       in_dir="/path/to/your/dataset/geneformer", 
                       tok_dir="/path/to/your/dataset/geneformer_token", 
                       prefix="processed"):
    os.makedirs(in_dir, exist_ok=True)
    h5ad_path = os.path.join(in_dir, f"{prefix}.h5ad")
    adata_min.write_h5ad(h5ad_path, compression="gzip")

    os.environ.setdefault("TRANSFORMERS_NO_TF", "1")
    os.environ.setdefault("TRANSFORMERS_NO_FLAX", "1")

    from geneformer.tokenizer import TranscriptomeTokenizer
    tk = TranscriptomeTokenizer(model_version=model_version, nproc=4,
                                custom_attr_name_dict={"barcode":"barcode"})
    os.makedirs(tok_dir, exist_ok=True)
    tk.tokenize_data(
        data_directory=in_dir,
        output_directory=tok_dir,
        output_prefix=prefix,
        file_format="h5ad"
    )
    dataset_path = os.path.join(tok_dir, f"{prefix}.dataset")
    return dataset_path

dataset_path = write_and_tokenize(adata_min, 
                                  in_dir="/path/to/your/st/your_wsi_id",
                                 tok_dir="/path/to/your/dataset/geneformer_token/your_wsi_id")

In [ ]:
def extract_gene_embeddings(dataset_path, 
                            ckpt_dir, 
                            model_version="V2",
                            out_dir="/path/to/your/dataset/geneformer_embedding", 
                            out_prefix="gene",
                            emb_layer=-1):
    from geneformer.emb_extractor import EmbExtractor
    os.makedirs(out_dir, exist_ok=True)
    embex = EmbExtractor(
        model_type="Pretrained", 
        num_classes=0,
        emb_mode="cell",              
        gene_emb_style="mean_pool",   
        max_ncells=100000,
        emb_layer=emb_layer,        
        model_version=model_version,
        forward_batch_size=1,
        nproc=0
    )
    embex.extract_embs(ckpt_dir, dataset_path, out_dir, out_prefix)
    gene_csv = os.path.join(out_dir, f"{out_prefix}.csv")
    return gene_csv

cell_csv = extract_gene_embeddings(
    dataset_path, 
    ckpt_dir='/path/to/your/Geneformer/Geneformer-V2-316M', 
    out_dir="/path/to/your/dataset/geneformer_embedding", 
    out_prefix=wsi_id,
    emb_layer=-1)

In [ ]:
ds = load_from_disk(dataset_path)
bar_ds = ds["barcode"] if "barcode" in ds.features else [str(i) for i in range(len(ds))]
print(len(bar_ds))
cell_embedding = pd.read_csv(cell_csv, index_col=0)
embed_size = cell_embedding.shape[-1]
print(embed_size)
cell_embedding.index = bar_ds
cell_embedding.to_csv(cell_csv)

In [ ]:
# -------- load model --------

model_version = "V2"  
ckpt_dir = '/path/to/your/Geneformer/Geneformer-V2-316M'  
gene_embeddings_npy = "/path/to/your/dataset/geneformer_embedding/your_wsi_id_gene_embeddings.npy"
gene_mask_npy = "/path/to/your/dataset/geneformer_embedding/your_wsi_id_gene_mask.npy"
os.makedirs(os.path.dirname(gene_embeddings_npy), exist_ok=True)

device = "cuda" if torch.cuda.is_available() else "cpu"
model = AutoModel.from_pretrained(ckpt_dir)
model.to(device).eval()

In [ ]:
tk = TranscriptomeTokenizer(model_version=model_version)
gene_token_dict = tk.gene_token_dict                     # {ensembl: token_id}
inv_gene_token = {v: k.split(".")[0] for k, v in gene_token_dict.items()}  # token_id -> ensembl(no version

from tqdm import tqdm
cell_id_field = "barcode" if "barcode" in ds.features else None
batch_size = 1
records = []
def as_tensor(x):
    return torch.tensor(x, dtype=torch.long, device=device)

gene_embeddings = np.zeros([len(ds), len(save_gene_list), embed_size])
gene_mask = np.zeros([len(ds), len(save_gene_list)])

for idx in tqdm(range(0, len(ds))):
    batch = ds[idx:idx+1]
    input_ids = as_tensor(batch["input_ids"])

    with torch.no_grad():
        outputs = model(input_ids=input_ids, output_hidden_states=True)
        h = outputs.hidden_states[-1]     # [B, L, D]
        h = h.detach().cpu().numpy()

    B, L, D = h.shape

    cell_id = batch[cell_id_field][0]

    ids = input_ids[0].cpu().tolist()

    for j in range(L):
        tok = ids[j]
        gene_id = inv_gene_token.get(tok, None)
        if gene_id in geneformer_ens_to_symbol.keys():
            symbol = geneformer_ens_to_symbol[gene_id]
            symbol_index = save_gene_list.index(symbol)
            gene_embeddings[idx, symbol_index, :] = h[0, j, :]
            gene_mask[idx, symbol_index] = 1

In [ ]:
np.save(gene_embeddings_npy, gene_embeddings)
np.save(gene_mask_npy, gene_mask)